# PyAegean in Five Minutes (100% Offline)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ryanpavlicek/pyaegean/blob/main/notebooks/quickstart-offline.ipynb)

A short, self-contained tour of **pyaegean** using only the zero-dependency core and bundled offline data. Every cell runs locally in seconds with no network access, no API keys, and no large model downloads.

You will run:

1. **Linear A** corpus inspection, wildcard sign search, and `KU-RO` accounting reconciliation.
2. **Cypriot syllabary** editorial apparatus (`IG XV 1`) and the source-cited Greek bridge.
3. **Greek NLP** (Beta Code, rule-based `pipeline`, named sentence policies, and sandhi).
4. **Quantitative scansion** (`score_scansion` across epic, tragedy, and prose controls).
5. **Biblical Koine** (bundled John 1, Genesis 1 LXX, and offline Dodson Strong's glosses).
6. **Verified gazetteer** (`aegean.geo`) and launching the local visual corpus browser (`aegean browse`).

In [ ]:
try:
    import aegean
except ImportError:
    %pip install -q pyaegean
    import aegean

print("pyaegean", aegean.__version__)
for cid in ("lineara", "linearb", "cypriot", "cyprominoan", "greek"):
    corpus = aegean.load(cid)
    print(f"  {cid:12} {len(corpus):5d} bundled documents")

## 1. Linear A: tablets, sign patterns, and accounting totals

Linear A is undeciphered; transliteration values are conventional (via Linear B). Structural checks like `KU-RO` ledger summation operate directly on numerals and fractions.

In [ ]:
from aegean.analysis import balance_check, word_matches_sign_pattern

la = aegean.load("lineara")
ht13 = la.get("HT13")
print(ht13.id, ht13.meta.site, ht13.meta.period)
print("First 8 tokens:", [t.text for t in ht13.tokens][:8])

checks = balance_check(ht13)
for chk in checks:
    print(f"Total {chk.marker}: stated={chk.stated_total}, computed={chk.computed_sum}, balances={chk.balances}")

matches = [w for w, _ in la.word_frequencies() if word_matches_sign_pattern(w, "KU-*-RO")]
print("Wildcard KU-*-RO:", matches)
assert "KU-MA-RO" in matches

## 2. Cypriot syllabary: editorial apparatus & source-cited Greek bridge

The bundled *Inscriptiones Graecae* XV 1 corpus preserves per-token reading certainty (`certain`, `unclear`, `restored`) and bridges Arcado-Cypriot Greek forms to Classical lemmas while keeping Eteocypriot Amathus texts cleanly unbridged.

In [ ]:
from aegean.scripts.cypriot import lexicon_entry
from aegean.scripts.cypriot.analysis import bridge_coverage

cyp = aegean.load("cypriot")
doc = cyp.get("IG XV 1, 95")
print(doc.id, "|", doc.meta.site)
for tok in doc.words:
    entry = lexicon_entry(tok.text)
    if entry is not None:
        print(f"  {tok.text:18} [{tok.status.value:8}] -> {entry.lemma:12} ({entry.gloss})  [{entry.attestation}]")

cov = bridge_coverage(cyp)
print(f"Bridge coverage: {cov.read_tokens}/{cov.word_tokens} tokens ({cov.coverage_pct:.1f}%), {cov.distinct_read_forms} distinct forms")

## 3. Ancient Greek NLP: offline baseline, sentence policies, and sandhi

The zero-dependency Greek pipeline gives immediate normalization, Beta Code conversion, source-preserving sentence segmentation (`prose`, `verse`, `inscription`, `papyrus`), crasis resolution, and baseline POS/lemmatization.

In [ ]:
from aegean import greek

print("Beta Code:", greek.betacode_to_unicode("mh=nin a)/eide qea\\ *phlhi+a/dew *a)xilh=os"))

records = greek.pipeline("Ἐν ἀρχῇ ἦν ὁ λόγος.")
for r in records:
    print(f"  {r.text:10} {r.upos:6} lemma={r.lemma:10} source={r.lemma_source.value}")

crasis = greek.resolve_sandhi("κἀγὼ")
print("Sandhi κἀγὼ ->", crasis.words, f"(kind={crasis.kind})")

pap = greek.segment_text("[λόγος.] καί.", policy="papyrus")
print("Papyrus policy sentences:", len(pap.sentences), [s for s in pap.sentences])

## 4. Quantitative metrical scansion & scoring

`greek.score_scansion` diagnoses hexameter, elegiac pentameter, iambic trimeter, and seven aeolic metres without raising on non-fitting prose, and records caesura, dactyl ratio, and lexical overrides (synizesis / long-by-nature).

In [ ]:
samples = [
    ("hexameter", "μῆνιν ἄειδε θεὰ Πηληϊάδεω Ἀχιλῆος"),
    ("trimeter", "ὦ κοινὸν αὐτάδελφον Ἰσμήνης κάρα"),
    ("hexameter", "τοῦτο πεζός ἐστι λόγος"),
]
for meter, line in samples:
    sc = greek.score_scansion(line, meter=meter)
    if sc.scans and sc.scansion is not None:
        print(
            f"{meter:10} OK   {sc.scansion.pattern}  "
            f"caesura={sc.scansion.caesura}  synizesis={list(sc.synizesis_used)}"
        )
    else:
        print(f"{meter:10} NO   ({sc.syllable_count} syls) — {sc.error}")

## 5. Biblical & Koine Greek (bundled offline samples)

John 1, Philemon 1, Genesis 1:1–5 (LXX), Psalm 22 LXX, and the 5,467-entry Dodson Koine lexicon ship inside the core wheel.

In [ ]:
from aegean.greek import bible

greek.use_dodson()
john = bible.load_nt("John", ref="1.1-1.2")
for doc in john:
    for i, idxs in enumerate(doc.lines, 1):
        print(f"{doc.id}:{i}  {' '.join(doc.tokens[k].text for k in idxs)}")

gen = bible.load_lxx("Gen 1:1")
for doc in gen:
    for i, idxs in enumerate(doc.lines, 1):
        print(f"LXX {doc.id}:{i}  {' '.join(doc.tokens[k].text for k in idxs)}")

entry = greek.lookup_nt("λόγος")
assert entry is not None
print(f"Dodson {entry.lemma} (G{entry.strongs}): {entry.gloss}")

## 6. Verified gazetteer & local visual Greek corpus browser

`aegean.geo` ships a Pleiades-aligned Mediterranean and Aegean gazetteer with spatial precision metadata and an `unmapped_sites` audit helper. For interactive reading, `aegean browse` (or `aegean.browser.serve()`) launches a zero-dependency local web reader with interlinear glosses, morphology, Strong's/LSJ lexica, patristic commentaries, and the Library Manager.

In [ ]:
from aegean import geo

coords = geo.site_coordinates()
aligned = sum(1 for c in coords.values() if c.pleiades)
print(f"Gazetteer: {len(coords)} sites ({aligned} Pleiades-aligned)")
print("Kurion:", coords["Kurion"])
print("Unmapped Cypriot sites:", geo.unmapped_sites(aegean.load("cypriot")))

# On a local machine (outside headless notebook CI), launch the interactive web browser:
#   from aegean import browser
#   browser.serve("John 1:1-18", open_browser=True)
# Or from the shell:
#   aegean browse "John 1:1-18"

## Next steps

- **Full guided tour:** [`getting-started.ipynb`](https://github.com/ryanpavlicek/pyaegean/blob/main/notebooks/getting-started.ipynb)
- **Specialist walkthroughs:** [`walkthrough-aegean.ipynb`](https://github.com/ryanpavlicek/pyaegean/blob/main/notebooks/walkthrough-aegean.ipynb), [`walkthrough-epigraphist.ipynb`](https://github.com/ryanpavlicek/pyaegean/blob/main/notebooks/walkthrough-epigraphist.ipynb), [`walkthrough-nt-koine.ipynb`](https://github.com/ryanpavlicek/pyaegean/blob/main/notebooks/walkthrough-nt-koine.ipynb)
- **Local visual browser:** [Greek Browser guide](https://github.com/ryanpavlicek/pyaegean/wiki/Greek-Browser)
- **Documentation & API reference:** [pyaegean.xyz](https://pyaegean.xyz/) · [Project Wiki](https://github.com/ryanpavlicek/pyaegean/wiki)